# BT4012 Bitcoin Illicit Transaction Detection - Final Model

This notebook reproduces **both submissions selected for the final private leaderboard** end to end: data
preprocessing, feature engineering, temporal validation, and generating the `.csv` submissions.

| Submission | Model | Public LB | Output file |
|---|---|---|---|
| 1 | Two-stage graph LightGBM (15 leaves) | **0.96032** | `submissions/e1249331_sub1.csv` |
| 2 | Same pipeline, larger stage-2 trees (31 leaves, 60% of columns, min 100 per leaf) | 0.95680 | `submissions/e1249331_sub2.csv` |

 It is self-contained (no imports from `src/`).

The model is a 2-stage LightGBM. 
- Stage 1 scores every transaction from its own features
- Stage 2 sees how suspicious each transaction's *labelled neighbours* look (mean/max of their stage-1 scores) and
structural counts of labelled vs unlabelled neighbours, and is trained with recency weighting (older time steps count less).

The full reasoning that led here (EDA, validation design, baselines, dropped ideas) is in `notebooks 01`–`06`.

**Setup:** put the Kaggle files in `data/raw/` (`train.csv`, `test.csv`, `txs_edgelist.csv`, `sample_submission.csv`)
and install `requirements.txt`. Runtime is roughly 10 minutes on a laptop, most of it in Section 4 (validation).

In [1]:
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
OUT = ROOT / "submissions"
OUT.mkdir(exist_ok=True)

FEATURE_COLS = [f"feat_{i}" for i in range(1, 166)]
SEEDS = (42, 1, 2, 3, 4)          # final model averages 5 seeds
DECAY = 0.95                      # recency weighting
LGB_PARAMS = dict(n_estimators=500, learning_rate=0.02, num_leaves=15, colsample_bytree=0.3,
                  class_weight="balanced", verbose=-1, n_jobs=8)

# Submission 2 only changes the stage-2 (final) model; stage 1 is identical.
SUB2_STAGE2_PARAMS = dict(num_leaves=31, colsample_bytree=0.6, min_child_samples=100)

## 1. Data loading and preprocessing

- `train.csv`: time steps 1–35; `label` is 1 (illicit), 0 (licit) or missing (unknown).
- `test.csv`: time steps 36–49, to be scored.
- `txs_edgelist.csv`: directed payment-flow edges (an output of `txId1` is spent by `txId2`).

Not much preprocessing to do since the 165 features are already numeric and scaled: we keep only labelled rows for
training and cast the label to int. Tree models need no scaling or imputation. `time_step` is **never** a model
input; it is used only to split validation folds and to set the recency weights.

In [2]:
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
edges = pd.read_csv(RAW / "txs_edgelist.csv")

L = train[train.label.notna()].copy()
L["label"] = L.label.astype(int)

print(f"train rows {len(train):,} (labelled {len(L):,}, illicit {L.label.mean():.1%}) | test rows {len(test):,} | edges {len(edges):,}")
assert test.isna().sum().sum() == 0 and L[FEATURE_COLS].isna().sum().sum() == 0

train rows 141,772 (labelled 31,235, illicit 11.7%) | test rows 15,329 | edges 234,355


## 2. Key observation behind the graph features

**Who has data.** The original Elliptic dataset had *labelled* transactions (licit or illicit) and *unknown* ones.
In this competition, all the labelled ones are in our files (train steps 1–35 plus the test set). The unknown ones
from the test period (steps 36–49) were removed and only appear in the edge list, as connections with no data attached.

**Why we only use labelled neighbours.** For a test transaction, its unknown neighbours have no features, so we cannot
look at them. In training, unknown neighbours *do* have features, but using them there would teach the model to rely on
information it will not have at test time. So in both train and test we only look at labelled neighbours e.g. a feature like
"average suspicion of my neighbours" is then calculated the same way in both.

**Why neighbours help.** Transactions only connect to others in the same time step, and illicit transactions mostly
connect to other illicit ones (the cell below shows about 57% of an illicit transaction's labelled neighbours are
illicit, against about 3% for a licit one). So if a transaction's neighbours look suspicious, it probably is too.
The neighbour-score features use exactly that.

In [3]:
known = set(L.txId) | set(test.txId)
edge_nodes = set(edges.txId1) | set(edges.txId2)
print(f"edge-only (unlabelled, featureless) nodes: {len(edge_nodes - set(train.txId) - set(test.txId)):,}")

lab = L.set_index("txId").label
e = edges.assign(a=edges.txId1.map(lab), b=edges.txId2.map(lab)).dropna()
pairs = pd.concat([e[["a", "b"]], e[["b", "a"]].set_axis(["a", "b"], axis=1)])
print("share of labelled neighbours that are illicit, by own class:")
print(pairs.groupby("a").b.mean().rename({0.0: "licit", 1.0: "illicit"}).round(3).to_string())

edge-only (unlabelled, featureless) nodes: 46,668
share of labelled neighbours that are illicit, by own class:
a
licit      0.028
illicit    0.574


## 3. Feature engineering

1. **Raw features:** the 165 given features.
2. **Structural features:** in/out degree from the full edge list, split into labelled and unlabelled neighbours,
   plus the share of neighbours that are labelled.
3. **Neighbour-score features:** a stage-1 LightGBM scores every transaction; each transaction then gets the mean
   and max stage-1 score over its incoming, outgoing and all labelled neighbours (6 features).

**Avoiding leakage in stage 1.** Training rows get *out-of-fold* scores: the training steps are cut into 5 blocks
of consecutive time steps, and each block is scored by a model trained on the other four. Because edges never cross
time steps, a transaction and all its neighbours are scored by a model that saw none of them. Rows being predicted
(validation or test) are scored by a stage-1 model trained on all training rows.

In [4]:
def structural_features(ids):
    e = edges.assign(k1=edges.txId1.isin(known), k2=edges.txId2.isin(known))
    f = pd.DataFrame(index=pd.Index(ids, name="txId"))
    f["indeg"] = e.txId2.value_counts()
    f["outdeg"] = e.txId1.value_counts()
    f["in_known"] = e[e.k1].txId2.value_counts()
    f["out_known"] = e[e.k2].txId1.value_counts()
    f = f.fillna(0)
    f["in_unk"] = f.indeg - f.in_known
    f["out_unk"] = f.outdeg - f.out_known
    f["frac_known"] = (f.in_known + f.out_known) / (f.indeg + f.outdeg).clip(lower=1)
    return f


kedges = edges[edges.txId1.isin(known) & edges.txId2.isin(known)]   # edges between labelled nodes


def neighbour_score_features(score, prefix="nb"):
    '''Mean / max of a per-node score over in-, out- and all labelled neighbours.'''
    out = {}
    for name, src, dst in [("in", "txId1", "txId2"), ("out", "txId2", "txId1")]:
        g = kedges[[src, dst]].assign(p=kedges[src].map(score)).dropna().groupby(dst).p
        out[f"{prefix}_{name}_mean"], out[f"{prefix}_{name}_max"] = g.mean(), g.max()
    both = pd.concat([kedges.assign(node=kedges.txId2, p=kedges.txId1.map(score)),
                      kedges.assign(node=kedges.txId1, p=kedges.txId2.map(score))]).dropna(subset=["p"])
    g = both.groupby("node").p
    out[f"{prefix}_all_mean"], out[f"{prefix}_all_max"] = g.mean(), g.max()
    return pd.DataFrame(out).reindex(score.index)


def fit_predict(Xf, yf, Xp, seeds=SEEDS, w=None, params=None):
    '''Seed-averaged LightGBM probabilities. `params` overrides LGB_PARAMS (used for submission 2's stage 2).'''
    P = {**LGB_PARAMS, **(params or {})}
    return np.mean([lgb.LGBMClassifier(**P, random_state=s).fit(Xf, yf, sample_weight=w).predict_proba(Xp)[:, 1]
                    for s in seeds], axis=0)


def stage1(fit, pred, feats):
    '''Out-of-fold scores for fit rows (5 blocks of consecutive steps); full-model scores for pred rows.'''
    s = pd.Series(index=pd.concat([fit.txId, pred.txId]).values, dtype=float)
    for block in np.array_split(np.array(sorted(fit.time_step.unique())), 5):
        m = fit.time_step.isin(block)
        s.loc[fit.txId[m].values] = fit_predict(fit.loc[~m, feats], fit.label[~m], fit.loc[m, feats], seeds=(42,))
    s.loc[pred.txId.values] = fit_predict(fit[feats], fit.label, pred[feats], seeds=(42,))
    return s


S = structural_features(pd.concat([L.txId, test.txId]))
STRUCT = list(S.columns)
L, test_f = L.join(S, on="txId"), test.join(S, on="txId")
S.loc[L.txId].describe().T[["mean", "50%", "max"]].round(2)

,mean,50%,max
indeg,1.76,1.00,284.0
outdeg,1.10,1.00,472.0
in_known,0.77,0.00,61.0
out_known,0.77,1.00,452.0
in_unk,1.00,0.00,273.0
out_unk,0.34,0.00,44.0
frac_known,0.63,0.75,1.0


## 4. Model and temporal validation

**Stage 2** is LightGBM on raw + structural + neighbour-score features, with **recency weighting**: a row from *k*
steps before the last training step gets weight 0.95<sup>*k*</sup>, so recent behaviour leads (illicit patterns
drift over time and the test steps follow step 35).

**Validation is temporal only:** fit on steps 1..*t*, score on steps *t*+1..35, pooled ROC-AUC (as Kaggle computes it).

- **Main check:** mean pooled AUC over *t* = 24, 26, 28, 30. In earlier notebooks this ordering of models agreed
  with the public leaderboard.
- **Sanity check:** mean over *t* = 20, 22, which hides an unusual illicit pattern first seen at step 23.

We compare the final model with the previous best (LightGBM on the 165 raw features, public LB 0.948).
3 seeds per split to keep runtime down.

In [5]:
def two_stage(fit, pred, decay=DECAY, seeds=SEEDS, stage2_params=None):
    '''Full pipeline: stage-1 OOF scores -> neighbour-score features -> recency-weighted stage 2.'''
    feats = FEATURE_COLS + STRUCT
    N = neighbour_score_features(stage1(fit, pred, feats))
    fit, pred = fit.join(N, on="txId"), pred.join(N, on="txId")
    feats = feats + list(N.columns)
    w = None if decay is None else decay ** (fit.time_step.max() - fit.time_step.values)
    return fit_predict(fit[feats], fit.label, pred[feats], seeds=seeds, w=w, params=stage2_params)


EASY, HARD = (24, 26, 28, 30), (20, 22)
rows = []
for t in HARD + EASY:
    fit, val = L[L.time_step <= t], L[L.time_step > t]
    base = fit_predict(fit[FEATURE_COLS], fit.label, val[FEATURE_COLS], seeds=(42, 1, 2))
    final = two_stage(fit, val, seeds=(42, 1, 2))
    rows.append({"fit steps": f"1-{t}", "check": "main" if t in EASY else "sanity",
                 "LightGBM raw 165": roc_auc_score(val.label, base), "final model": roc_auc_score(val.label, final)})
res = pd.DataFrame(rows)
display(res.round(4))
res.groupby("check")[["LightGBM raw 165", "final model"]].mean().round(4)

,fit steps,check,LightGBM raw 165,final model
0,1-20,sanity,0.9066,0.9088
1,1-22,sanity,0.8788,0.8763
2,1-24,main,0.9972,0.9979
3,1-26,main,0.9975,0.9985
4,1-28,main,0.9972,0.9985
5,1-30,main,0.9985,0.9986


,LightGBM raw 165,final model
check,,
main,0.9976,0.9984
sanity,0.8927,0.8925


The final model improves the main check and holds up on the sanity check. The gains look small because pooled AUC
is already near 0.998 on these splits; on the public leaderboard the same change moved the score from 0.948 to
**0.960**. Submission 2 (larger stage-2 trees) tied it on validation in `scripts/validate_lgb_params.py` (main check
0.9983, sanity check 0.8978) and was selected as a second, slightly different model; it scored 0.95680 on the public
leaderboard. Ideas tested and dropped (second round of neighbour scores, two-hop scores, ExtraTrees/RF blends,
post-hoc neighbour smoothing, stronger decay, other LightGBM settings) are documented in `notebooks/06_graph_model.ipynb`
and `experiments.csv`.

## 5. Final fit on steps 1–35 and both submissions

Refit the full pipeline on all labelled rows of steps 1–35 (stage 2 averages 5 seeds), convert the scores to ranks
scaled to [0, 1] (AUC depends only on the ranking), check each file against `sample_submission.csv`, and save it.
The two submissions share the same features and stage 1; only the stage-2 LightGBM settings differ.

In [6]:
sample = pd.read_csv(RAW / "sample_submission.csv")


def make_submission(stage2_params, filename):
    '''Refit on steps 1-35, rank-scale the scores to [0, 1], validate the format and save.'''
    p = rankdata(two_stage(L, test_f, stage2_params=stage2_params))
    p = (p - p.min()) / (p.max() - p.min())
    sub = pd.DataFrame({"index": test["index"].values, "target": p})
    assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
    assert (sub["index"].values == sample["index"].values).all()
    assert sub.target.between(0, 1).all() and sub.target.notna().all()
    sub.to_csv(OUT / filename, index=False)
    print("saved", OUT / filename, sub.shape)
    return sub


sub1 = make_submission(None, "e1249331_sub1.csv")                 # public LB 0.96032
sub2 = make_submission(SUB2_STAGE2_PARAMS, "e1249331_sub2.csv")   # public LB 0.95680
sub1.head()

saved /Users/chloe/bt4012/submissions/e1249331_sub1.csv (15329, 2)


saved /Users/chloe/bt4012/submissions/e1249331_sub2.csv (15329, 2)


,index,target
0,0,0.550626
1,1,0.695068
2,2,0.587748
3,3,0.525835
4,4,0.343293


In [7]:
# Check: each file reproduces the one submitted to Kaggle (if the originals are present).
for sub, original, lb in [(sub1, "2026-10-23_lgb_graph_nb_decay95.csv", 0.96032),
                          (sub2, "2026-10-24_lgb_graph_nb_decay95_31leaves.csv", 0.95680)]:
    if (OUT / original).exists():
        ref = pd.read_csv(OUT / original)
        print(f"LB {lb}: max abs difference vs submitted file = {np.abs(ref.target.values - sub.target.values).max():.2e}")

LB 0.96032: max abs difference vs submitted file = 1.11e-16
LB 0.9568: max abs difference vs submitted file = 1.11e-16


## Summary

| Submission | Public LB |
|---|---|
| Random Forest, 165 raw features | 0.934 |
| LightGBM, 165 raw features | 0.948 |
| **Submission 1: two-stage graph LightGBM + recency weighting (this notebook)** | **0.960** |
| Submission 2: same, larger stage-2 trees (this notebook) | 0.957 |

What made the difference: (1) noticing that the test set is exactly the originally labelled transactions, so
labelled-neighbour features are consistent between train and test; (2) using neighbours' *predicted scores* (with
out-of-fold stage-1 scores to avoid leakage) rather than their raw features; (3) weighting recent time steps more.